# CMP611 Final Epoch-Based Low-Data Run

This notebook reproduces the final DNABERT-2 low-data fine-tuning experiments.

Runtime requirement: use **Colab T4 GPU**. The full grid should not be run on CPU.

Recommended workflow: run one seed at a time and download its result archive immediately.


In [ ]:
# 0) Clone the project repository
import os, shutil, subprocess

REPO_URL = 'https://github.com/FerasAlaqad/CMP611-Low-Data-Fine-Tuning-Analysis-of-DNABERT-2-.git'
PROJECT_DIR = '/content/project'

if os.path.exists(PROJECT_DIR):
    shutil.rmtree(PROJECT_DIR)

subprocess.check_call(['git', 'clone', REPO_URL, PROJECT_DIR])
print('Project directory:', PROJECT_DIR)
print('Files:', sorted(os.listdir(PROJECT_DIR))[:30])


In [ ]:
# 1) Install stable dependencies
%cd /content/project

!pip -q install -r requirements.txt
!pip -q install genomic-benchmarks==0.0.9


In [ ]:
# 2) Runtime, environment variables, and CUDA sanity check
%cd /content/project

import os
import platform
import subprocess
import sys

# Environment variables used during the run.
# These keep logging quiet, avoid tokenizer deadlock warnings, and make CUDA allocation safer.
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['WANDB_DISABLED'] = 'true'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ.setdefault('HF_HOME', '/content/hf_cache')
os.environ.setdefault('TRANSFORMERS_CACHE', '/content/hf_cache')
os.environ.setdefault('HF_HUB_DISABLE_TELEMETRY', '1')

print('=== Runtime ===')
print('Python:', sys.version.split()[0])
print('Platform:', platform.platform())
print('Working directory:', os.getcwd())
print('\n=== Environment variables ===')
for key in [
    'TOKENIZERS_PARALLELISM',
    'WANDB_DISABLED',
    'PYTORCH_CUDA_ALLOC_CONF',
    'HF_HOME',
    'TRANSFORMERS_CACHE',
    'HF_HUB_DISABLE_TELEMETRY',
    'CUDA_VISIBLE_DEVICES',
]:
    print(f'{key}={os.environ.get(key, "<not set>")}')

print('\n=== nvidia-smi ===')
nvidia = subprocess.run(['bash', '-lc', 'nvidia-smi'], text=True, capture_output=True)
print(nvidia.stdout if nvidia.stdout.strip() else nvidia.stderr)
GPU_VISIBLE_TO_SYSTEM = (nvidia.returncode == 0)

print('\n=== PyTorch CUDA check ===')
import torch
print('torch:', torch.__version__)
print('torch cuda build:', torch.version.cuda)
print('cuda available:', torch.cuda.is_available())
print('device count:', torch.cuda.device_count())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

if not GPU_VISIBLE_TO_SYSTEM:
    raise RuntimeError(
        'No GPU is visible to the Colab runtime. Select a T4 GPU runtime and restart the notebook.'
    )

if GPU_VISIBLE_TO_SYSTEM and not torch.cuda.is_available():
    print('\nGPU is visible to nvidia-smi, but PyTorch cannot use CUDA.')
    print('Installing a CUDA-enabled PyTorch wheel. Restart the runtime after this cell finishes.')
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
        'torch', 'torchvision', 'torchaudio',
        '--index-url', 'https://download.pytorch.org/whl/cu121'
    ])
    raise RuntimeError(
        'CUDA PyTorch was reinstalled. Restart the Colab runtime and run the notebook again.'
    )

print('\n=== Project environment check ===')
subprocess.run([sys.executable, 'scripts/check_environment.py'], check=True)
print('\nEnvironment is OK. GPU is visible and fp16 training is safe.')


In [ ]:
# 3) Download GUE
%cd /content/project
!python scripts/download_gue.py --out-dir data/raw

In [ ]:
# 4) Build GUE low-data splits: 0.05%, 0.1%, 1%, 10%, 100%
%cd /content/project

RATIOS = '0.0005,0.001,0.01,0.10,1.0'
SEEDS = '13,42,3407'

!python scripts/build_low_data_splits.py \
  --source-dir data/raw/GUE/prom/prom_300_all \
  --output-root data/low_data/final_epoch/prom_300_all \
  --ratios $RATIOS \
  --seeds $SEEDS

!python scripts/build_low_data_splits.py \
  --source-dir data/raw/GUE/prom/prom_core_all \
  --output-root data/low_data/final_epoch/prom_core_all \
  --ratios $RATIOS \
  --seeds $SEEDS

!python scripts/build_low_data_splits.py \
  --source-dir data/raw/GUE/splice/reconstructed \
  --output-root data/low_data/final_epoch/splice_reconstructed \
  --ratios $RATIOS \
  --seeds $SEEDS

In [ ]:
# 5) Download + convert Non-TATA promoter dataset
%cd /content/project

from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split
from genomic_benchmarks.loc2seq import download_dataset

dest = Path('/content/gb_data')
dest.mkdir(parents=True, exist_ok=True)
ret = download_dataset('human_nontata_promoters', version=0, dest_path=dest, force_download=False)

candidates = []
if ret is not None:
    candidates.append(Path(ret))
candidates.append(dest / 'human_nontata_promoters')
root = next((p for p in candidates if p.exists()), None)
if root is None:
    raise FileNotFoundError('human_nontata_promoters was downloaded, but the folder was not found.')

print('Dataset root:', root)

def load_split(split_name):
    split_dir = root / split_name
    class_dirs = sorted([d for d in split_dir.iterdir() if d.is_dir()])
    label_map = {d.name: i for i, d in enumerate(class_dirs)}
    rows = []
    for cls_dir in class_dirs:
        label_id = label_map[cls_dir.name]
        for fp in cls_dir.glob('*.txt'):
            seq = fp.read_text().strip().upper()
            if seq:
                rows.append({'sequence': seq, 'label': label_id})
    return pd.DataFrame(rows), label_map

train_df, label_map = load_split('train')
test_df, _ = load_split('test')
train_df, dev_df = train_test_split(train_df, test_size=0.10, random_state=13, stratify=train_df['label'])

out = Path('/content/project/data/raw/extra/human_nontata_promoters')
out.mkdir(parents=True, exist_ok=True)
train_df.to_csv(out / 'train.csv', index=False)
dev_df.to_csv(out / 'dev.csv', index=False)
test_df.to_csv(out / 'test.csv', index=False)

print('Saved:', out)
print('Label map:', label_map)
print('train/dev/test:', len(train_df), len(dev_df), len(test_df))

In [ ]:
# 6) Build Non-TATA low-data splits
%cd /content/project

RATIOS = '0.0005,0.001,0.01,0.10,1.0'
SEEDS = '13,42,3407'

!python scripts/build_low_data_splits.py \
  --source-dir data/raw/extra/human_nontata_promoters \
  --output-root data/low_data/final_epoch/human_nontata_promoters \
  --ratios $RATIOS \
  --seeds $SEEDS

In [ ]:
# 7) Inspect split sizes before training
%cd /content/project
python - << 'PY'
from pathlib import Path
import json

for meta in sorted(Path('data/low_data/final_epoch').rglob('meta.json')):
    m = json.loads(meta.read_text())
    print(str(meta.parent), 'train=', m['train_size'], 'dev=', m['dev_size'], 'test=', m['test_size'], 'ratio=', m['ratio'], 'seed=', m['seed'])
PY

## Seed-By-Seed Final Runs

Run order:

1. Run all tasks for seed `13`, then immediately download `final_epoch_seed13_results_light.zip`.
2. Run all tasks for seed `42`, then immediately download `final_epoch_seed42_results_light.zip`.
3. Run all tasks for seed `3407`, then immediately download `final_epoch_seed3407_results_light.zip`.

Each seed has `4 tasks × 5 data ratios = 20 runs`.

If Colab disconnects during a seed, run the same seed cell again. Completed runs are skipped automatically when `eval_results.json` is present.


In [ ]:
# 8) Seed runner and result export
%cd /content/project

import json
import os
import shutil
import subprocess
import time
from pathlib import Path
from zipfile import ZIP_DEFLATED, ZipFile

import pandas as pd
import torch
from google.colab import files

# Final settings
MODEL = 'zhihan1996/DNABERT-2-117M'
BASE_OUT = Path('outputs/final_epoch/main_runs')
NUM_EPOCHS = 3

TASKS = [
    ('promoter',         'data/low_data/final_epoch/prom_300_all',            300),
    ('core_promoter',    'data/low_data/final_epoch/prom_core_all',            70),
    ('splice',           'data/low_data/final_epoch/splice_reconstructed',    400),
    ('nontata_promoter', 'data/low_data/final_epoch/human_nontata_promoters', 251),
]

# Final ratios: 0.05%, 0.1%, 1%, 10%, 100%.
RATIO_TAGS = ['r0p05', 'r0p1', 'r1', 'r10', 'r100']
RATIO_LABELS = {
    'r0p05': '0.05%',
    'r0p1': '0.1%',
    'r1': '1%',
    'r10': '10%',
    'r100': '100%',
}

REQUIRE_GPU_FOR_FULL_RUN = True
USE_CUDA = torch.cuda.is_available()
if REQUIRE_GPU_FOR_FULL_RUN and not USE_CUDA:
    raise RuntimeError('CUDA is not available. Fix Colab runtime before running final experiments.')

print('CUDA:', USE_CUDA, '| GPU:', torch.cuda.get_device_name(0) if USE_CUDA else 'CPU')
print('Runs per seed:', len(TASKS) * len(RATIO_TAGS))


def result_file_for(task_name, ratio_tag, seed):
    split = f'{ratio_tag}_seed{seed}'
    run_name = f'{task_name}_{ratio_tag}_seed{seed}'
    return BASE_OUT / task_name / split / 'results' / run_name / 'eval_results.json'


def read_result_summary(task_name, ratio_tag, seed):
    run_name = f'{task_name}_{ratio_tag}_seed{seed}'
    result_file = result_file_for(task_name, ratio_tag, seed)
    data = json.loads(result_file.read_text())
    return {
        'task': task_name,
        'run_name': run_name,
        'ratio_percent': RATIO_LABELS[ratio_tag].replace('%', ''),
        'seed': seed,
        'eval_f1_macro': data.get('eval_f1_macro'),
        'eval_auroc': data.get('eval_auroc'),
        'eval_auprc': data.get('eval_auprc'),
        'eval_accuracy': data.get('eval_accuracy'),
        'eval_matthews_correlation': data.get('eval_matthews_correlation'),
        'eval_runtime': data.get('eval_runtime'),
        'epoch': data.get('epoch'),
    }


def print_result_for_console(task_name, ratio_tag, seed):
    """Print completed run metrics."""
    summary = read_result_summary(task_name, ratio_tag, seed)
    print('\nRESULT_ONE_RUN_FOR_CONSOLE')
    print(json.dumps(summary, indent=2, sort_keys=True))
    print('RESULT_ONE_RUN_FOR_CONSOLE_END')
    print(
        'RESULT_CSV_LINE_FOR_CONSOLE,'
        f"{summary['task']},{summary['run_name']},{summary['ratio_percent']},{summary['seed']},"
        f"{summary['eval_f1_macro']},{summary['eval_auroc']},{summary['eval_auprc']},"
        f"{summary['eval_accuracy']},{summary['eval_matthews_correlation']}"
    )


def completed_runs_for_seed(seed):
    done = []
    for task_name, _, _ in TASKS:
        for ratio_tag in RATIO_TAGS:
            if result_file_for(task_name, ratio_tag, seed).exists():
                done.append((task_name, ratio_tag, seed))
    return done


def print_seed_status(seed):
    total = len(TASKS) * len(RATIO_TAGS)
    done = completed_runs_for_seed(seed)
    print(f'\nSeed {seed} progress: {len(done)}/{total} completed')
    for task_name, _, _ in TASKS:
        status = []
        for ratio_tag in RATIO_TAGS:
            mark = '✓' if result_file_for(task_name, ratio_tag, seed).exists() else '·'
            status.append(f'{RATIO_LABELS[ratio_tag]}:{mark}')
        print(f'  {task_name:16s} ' + ' | '.join(status))


def run_one(task_name, data_root, max_len, ratio_tag, seed):
    split = f'{ratio_tag}_seed{seed}'
    data_path = f'{data_root}/{split}'
    run_name = f'{task_name}_{ratio_tag}_seed{seed}'
    out_dir = BASE_OUT / task_name / split
    result_file = out_dir / 'results' / run_name / 'eval_results.json'

    if result_file.exists():
        print(f'\nSKIP existing completed run: {run_name}')
        print_result_for_console(task_name, ratio_tag, seed)
        return 'skipped'

    if out_dir.exists():
        shutil.rmtree(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    cmd = [
        'python', 'scripts/train_dnabert2.py',
        '--model_name_or_path', MODEL,
        '--data_path', data_path,
        '--run_name', run_name,
        '--output_dir', str(out_dir),
        '--seed', str(seed),
        '--model_max_length', str(max_len),
        '--num_train_epochs', str(NUM_EPOCHS),
        '--per_device_train_batch_size', '8',
        '--per_device_eval_batch_size', '16',
        '--gradient_accumulation_steps', '1',
        '--learning_rate', '3e-5',
        '--warmup_ratio', '0.06',
        '--weight_decay', '0.01',
        '--evaluation_strategy', 'epoch',
        '--save_strategy', 'no',
        '--logging_steps', '10',
        '--load_best_model_at_end', 'False',
        '--report_to', 'none',
    ]
    if USE_CUDA:
        cmd += ['--fp16', 'True']

    print('\n' + '=' * 100)
    print(f'RUNNING: seed={seed} | task={task_name} | ratio={RATIO_LABELS[ratio_tag]} | max_len={max_len}')
    print('COMMAND:', ' '.join(cmd))
    print('=' * 100)

    started = time.time()
    subprocess.run(cmd, check=True)
    elapsed = time.time() - started

    if not result_file.exists():
        raise RuntimeError(f'Run finished but result file was not created: {result_file}')

    print(f'FINISHED: {run_name} | elapsed={elapsed/60:.1f} min')
    print_result_for_console(task_name, ratio_tag, seed)
    return elapsed


def run_seed(seed):
    seed_started = time.time()
    total = len(TASKS) * len(RATIO_TAGS)
    completed_before = len(completed_runs_for_seed(seed))
    run_times = []

    print('\n' + '#' * 100)
    print(f'START SEED {seed}: {total} runs planned. Already completed in this runtime: {completed_before}/{total}')
    print('#' * 100)
    print_seed_status(seed)

    for task_name, data_root, max_len in TASKS:
        for ratio_tag in RATIO_TAGS:
            done_before = len(completed_runs_for_seed(seed))
            print(f'\nProgress before run: {done_before}/{total} completed for seed {seed}')

            elapsed = run_one(task_name, data_root, max_len, ratio_tag, seed)
            if isinstance(elapsed, (int, float)):
                run_times.append(elapsed)

            done_after = len(completed_runs_for_seed(seed))
            remaining = total - done_after
            avg = sum(run_times) / len(run_times) if run_times else 0
            eta = avg * remaining if avg else 0
            print(f'\nProgress after run: {done_after}/{total} completed for seed {seed}')
            if eta:
                print(f'Approx ETA for this seed from observed average: {eta/60:.1f} min')
            print_seed_status(seed)

    total_elapsed = time.time() - seed_started
    print('\n' + '#' * 100)
    print(f'SEED {seed} COMPLETE: {len(completed_runs_for_seed(seed))}/{total} runs | elapsed={total_elapsed/3600:.2f} hours')
    print('#' * 100)


def build_seed_summary(seed):
    subprocess.run([
        'python', 'scripts/aggregate_results.py',
        '--results-root', str(BASE_OUT),
        '--out-csv', 'outputs/final_epoch_summary_all_runs.csv',
        '--out-mean-csv', 'outputs/final_epoch_summary_mean_std.csv',
    ], check=True)

    all_csv = Path('outputs/final_epoch_summary_all_runs.csv')
    df = pd.read_csv(all_csv)
    seed_df = df[df['seed'].astype(str) == str(seed)].copy()
    if seed_df.empty:
        raise RuntimeError(f'No rows found for seed {seed}; cannot build seed summary.')

    seed_all = Path(f'outputs/final_epoch_seed{seed}_summary_all_runs.csv')
    seed_mean = Path(f'outputs/final_epoch_seed{seed}_summary_mean_std.csv')
    seed_df.to_csv(seed_all, index=False)

    metric_cols = [c for c in seed_df.columns if c.startswith('eval_')]
    mean_df = seed_df.groupby(['task', 'method', 'ratio_percent'])[metric_cols].agg(['mean', 'std']).reset_index()
    mean_df.columns = ['_'.join(col).strip('_') for col in mean_df.columns.values]
    mean_df.to_csv(seed_mean, index=False)

    simple_cols = ['task', 'run_name', 'ratio_percent', 'seed', 'eval_f1_macro', 'eval_auroc', 'eval_auprc']
    simple = seed_df[simple_cols].sort_values(['task', 'ratio_percent'])

    print('\nSeed summary files:')
    print(seed_all, 'rows=', len(seed_df))
    print(seed_mean, 'rows=', len(mean_df))
    print(simple.to_string(index=False))

    # Console-safe CSV block. If download fails, copy this block from output.
    print('\nSEED_SUMMARY_CSV_FOR_CONSOLE_START')
    print(simple.to_csv(index=False).strip())
    print('SEED_SUMMARY_CSV_FOR_CONSOLE_END')
    return seed_all, seed_mean


def zip_and_download_seed(seed):
    seed_all, seed_mean = build_seed_summary(seed)
    out = Path(f'/content/final_epoch_seed{seed}_results_light.zip')
    if out.exists():
        out.unlink()

    include = [seed_all, seed_mean, Path('outputs/final_epoch_summary_all_runs.csv'), Path('outputs/final_epoch_summary_mean_std.csv')]
    include.extend([p for p in BASE_OUT.rglob('eval_results.json') if f'seed{seed}' in str(p)])
    include.extend([p for p in Path('data/low_data/final_epoch').rglob('meta.json') if f'seed{seed}' in str(p)])

    with ZipFile(out, 'w', ZIP_DEFLATED) as z:
        for p in include:
            if p.exists():
                z.write(p, p.as_posix())

    print('\nCreated:', out)
    print('Files:', len(include))
    print('Size MB:', round(out.stat().st_size / 1024 / 1024, 3))
    print('If browser download fails, copy the SEED_SUMMARY_CSV_FOR_CONSOLE block above.')
    files.download(str(out))


In [ ]:
# 9A) Run seed 13 and download its result zip
run_seed(13)
zip_and_download_seed(13)


In [ ]:
# 9B) Run seed 42 and download its result zip
run_seed(42)
zip_and_download_seed(42)


In [ ]:
# 9C) Run seed 3407 and download its result zip
run_seed(3407)
zip_and_download_seed(3407)


In [ ]:
# 10) Aggregate all results currently present in this runtime
%cd /content/project
!python scripts/aggregate_results.py \
  --results-root outputs/final_epoch/main_runs \
  --out-csv outputs/final_epoch_summary_all_runs.csv \
  --out-mean-csv outputs/final_epoch_summary_mean_std.csv

import pandas as pd
for p in ['outputs/final_epoch_summary_all_runs.csv', 'outputs/final_epoch_summary_mean_std.csv']:
    df = pd.read_csv(p)
    print('\n', p, 'rows=', len(df))
    print(df.head(30).to_string(index=False))


In [ ]:
# 11) Create one combined archive from available results
%cd /content/project
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files

out = Path('/content/final_epoch_all_available_results_light.zip')
if out.exists():
    out.unlink()

include = []
for p in [
    Path('outputs/final_epoch_summary_all_runs.csv'),
    Path('outputs/final_epoch_summary_mean_std.csv'),
]:
    if p.exists():
        include.append(p)
include.extend(Path('outputs/final_epoch/main_runs').rglob('eval_results.json'))
include.extend(Path('data/low_data/final_epoch').rglob('meta.json'))

with ZipFile(out, 'w', ZIP_DEFLATED) as z:
    for p in include:
        z.write(p, p.as_posix())

print('Created:', out)
print('Files:', len(include))
print('Size MB:', round(out.stat().st_size / 1024 / 1024, 3))
files.download(str(out))
